# 2.6 注意力的复杂度 / Attention complexity

数清一层要做多少次乘法。输入变长时，哪部分线性增长，哪部分平方增长。

Count the multiplications in one layer, and see which part grows linearly and which quadratically as the input gets longer.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 计算并测量注意力的代价 / Compute and measure the cost of attention

写出一层中各部分的乘法次数、线性项与平方项相等时的长度，以及注意力矩阵占用的显存。再实现一个对数坐标下的斜率拟合，用它从实测时间里估计注意力耗时随长度增长的幂次。

Write the number of multiplications of each part of a layer, the length at which the linear and quadratic terms are equal, and the memory taken by the attention matrices. Then implement a slope fit in log coordinates and use it to estimate from measured times the power with which attention time grows with length.

In [ ]:
import math
import time
import torch


def layer_multiplications(T, d):
    """Scalar multiplications in ONE Transformer layer for a sequence of length T (biases, norms ignored).

    Returns a dict with integer entries:
      "projections": the four d x d matrices of attention (Q, K, V, output), applied to T positions
      "scores":      Q K^T over all heads together
      "mixing":      weights times values, over all heads together
      "mlp":         d -> 4d -> d, applied to T positions
      "linear":      the part that grows like T      (projections + mlp)
      "quadratic":   the part that grows like T^2    (scores + mixing)
    """
    # TODO 1
    raise NotImplementedError


def crossover_length(d):
    """The sequence length at which the quadratic part equals the linear part."""
    # TODO 2
    raise NotImplementedError


def attention_matrix_bytes(T, n_head, n_layer, bytes_per_number=2, batch=1):
    """Memory needed to hold the attention weights of every layer at once (float16 by default)."""
    # TODO 3
    raise NotImplementedError


def fit_exponent(xs, ys):
    """Least-squares slope of log(y) against log(x). If y = c * x^p this returns p."""
    # TODO 4: with u = log x and v = log y,  slope = sum((u - mean u)(v - mean v)) / sum((u - mean u)^2)
    raise NotImplementedError


def time_attention(T, d=64, repeats=3):
    """Seconds for one single-head attention over T positions (best of `repeats`). Nothing to change."""
    q, k, v = torch.randn(T, d), torch.randn(T, d), torch.randn(T, d)
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        torch.softmax(q @ k.T / math.sqrt(d), dim=-1) @ v
        best = min(best, time.perf_counter() - t0)
    return best

In [ ]:
# 检查 / checks
import math

c = layer_multiplications(1024, 768)
assert c["projections"] == 4 * 1024 * 768 ** 2 and c["mlp"] == 8 * 1024 * 768 ** 2
assert c["scores"] == c["mixing"] == 1024 ** 2 * 768
assert c["linear"] == 12 * 1024 * 768 ** 2 and c["quadratic"] == 2 * 1024 ** 2 * 768
assert c["linear"] > c["quadratic"], "at GPT-2 small's full context the linear part still dominates"

assert crossover_length(768) == 4608
x = layer_multiplications(crossover_length(768), 768)
assert x["linear"] == x["quadratic"]
big = layer_multiplications(100_000, 4096)
assert big["quadratic"] > 4 * big["linear"], "at 100k tokens the quadratic part dominates"

# doubling T doubles the linear part and quadruples the quadratic part
a, b = layer_multiplications(2000, 512), layer_multiplications(4000, 512)
assert b["linear"] == 2 * a["linear"] and b["quadratic"] == 4 * a["quadratic"]

assert attention_matrix_bytes(1024, 12, 12) == 12 * 12 * 1024 ** 2 * 2                 # GPT-2 small, about 0.3 GB
assert attention_matrix_bytes(8192, 32, 32) / 2 ** 30 == 128.0, "8k context, 32 heads, 32 layers: 128 GiB if stored naively"

assert abs(fit_exponent([1, 2, 4, 8], [3, 12, 48, 192]) - 2.0) < 1e-9
assert abs(fit_exponent([10, 100, 1000], [5, 50, 500]) - 1.0) < 1e-9

Ts = [512, 1024, 2048, 4096]
secs = [time_attention(T) for T in Ts]
p = fit_exponent(Ts, secs)
assert p > 1.5, f"measured exponent {p:.2f}: attention time should grow clearly faster than linearly"
print(f"m26_cost: all tests passed (measured time exponent {p:.2f})")

<details><summary><b>提示 1 / Hint 1</b></summary>

四项分别是 `4*T*d*d`、`T*T*d`、`T*T*d` 和 `8*T*d*d`。

The four terms are `4*T*d*d`, `T*T*d`, `T*T*d` and `8*T*d*d`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

拟合斜率时先对 `xs` 和 `ys` 取 `math.log`，再套一元线性回归的斜率公式。

For the slope take `math.log` of `xs` and `ys` first and apply the slope formula of simple linear regression.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import time
import torch


def layer_multiplications(T, d):
    projections = 4 * T * d * d          # each of T positions goes through four d x d matrices
    scores = T * T * d                   # h heads, each T x T dot products of size d/h
    mixing = T * T * d                   # h heads, each a (T x T) @ (T x d/h) product
    mlp = 2 * T * d * 4 * d              # up to 4d and back down
    return {"projections": projections, "scores": scores, "mixing": mixing, "mlp": mlp,
            "linear": projections + mlp, "quadratic": scores + mixing}


def crossover_length(d):
    return 6 * d                         # 2 T^2 d = 12 T d^2


def attention_matrix_bytes(T, n_head, n_layer, bytes_per_number=2, batch=1):
    return batch * n_layer * n_head * T * T * bytes_per_number


def fit_exponent(xs, ys):
    u = [math.log(x) for x in xs]
    v = [math.log(y) for y in ys]
    mu, mv = sum(u) / len(u), sum(v) / len(v)
    return sum((a - mu) * (b - mv) for a, b in zip(u, v)) / sum((a - mu) ** 2 for a in u)


def time_attention(T, d=64, repeats=3):
    q, k, v = torch.randn(T, d), torch.randn(T, d), torch.randn(T, d)
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        torch.softmax(q @ k.T / math.sqrt(d), dim=-1) @ v
        best = min(best, time.perf_counter() - t0)
    return best
```

</details>

## 2. 估算给定配置的计算量与显存 / Estimate compute and memory for a given configuration

这类估算题在面试中常见。设模型有 $L = 32$ 层，$d = 4096$，$h = 32$ 个头。
(a) 推导一层的乘法次数约为 $12 T d^2 + 2 T^2 d$。
(b) 序列多长时平方项与线性项相等？
(c) 输入 $T = 8192$ 个 token。若把所有层的注意力权重用半精度同时存下来，需要多少显存？只存一层呢？
(d) 不用 KV cache 生成 $n$ 个 token，总计算量随 $n$ 以什么幂次增长？使用 KV cache 后呢？

Estimates of this kind are common in interviews. Let the model have $L = 32$ layers, $d = 4096$ and $h = 32$ heads.
(a) Derive that one layer takes about $12 T d^2 + 2 T^2 d$ multiplications.
(b) At what sequence length are the quadratic and the linear term equal?
(c) The input has $T = 8192$ tokens. How much memory is needed to hold the attention weights of all layers at once in half precision? And of a single layer?
(d) Generating $n$ tokens without a KV cache, with what power of $n$ does the total computation grow? And with a KV cache?

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 分别数四个投影矩阵、分数矩阵、加权求和与 MLP。

(a) Count the four projection matrices, the score matrix, the weighted sum and the MLP separately.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(c) 每层有 $h \cdot T^2$ 个数，半精度每个数 2 字节，$1\ \text{GiB} = 2^{30}$ 字节。

(c) Each layer has $h \cdot T^2$ numbers, half precision uses 2 bytes per number, and $1\ \text{GiB} = 2^{30}$ bytes.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

(d) 第 $t$ 步的代价分别是 $a t + b t^2$ 和 $a + b t$，对 $t$ 从 1 到 $n$ 求和。

(d) The cost of step $t$ is $a t + b t^2$ and $a + b t$ respectively. Sum over $t$ from 1 to $n$.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 四个投影各 $T d^2$，共 $4 T d^2$。分数矩阵 $h \cdot T^2 \cdot d/h = T^2 d$。加权求和同样是 $T^2 d$。MLP 为 $T(4d^2 + 4d^2) = 8 T d^2$。合计 $12 T d^2 + 2 T^2 d$。

(b) $12 T d^2 = 2 T^2 d$ 给出 $T = 6d = 24576$。

(c) 一层是 $32 \cdot 8192^2 \cdot 2 = 2^5 \cdot 2^{26} \cdot 2 = 2^{32}$ 字节，即 4 GiB。32 层同时存是 128 GiB。推理时各层依次计算，不需要同时保存，实际实现还会分块计算而不存完整矩阵。

(d) 不用缓存时总量约为 $\tfrac{a}{2} n^2 + \tfrac{b}{3} n^3$，最高是三次方。使用 KV cache 后约为 $a n + \tfrac{b}{2} n^2$，最高是二次方。

---

(a) The four projections take $T d^2$ each, $4 T d^2$ in total. The score matrix takes $h \cdot T^2 \cdot d/h = T^2 d$ and the weighted sum another $T^2 d$. The MLP takes $T(4d^2 + 4d^2) = 8 T d^2$. Together $12 T d^2 + 2 T^2 d$.

(b) $12 T d^2 = 2 T^2 d$ gives $T = 6d = 24576$.

(c) One layer is $32 \cdot 8192^2 \cdot 2 = 2^5 \cdot 2^{26} \cdot 2 = 2^{32}$ bytes, that is 4 GiB. All 32 layers at once are 128 GiB. At inference the layers are computed one after another and need not be held together, and real implementations compute in blocks without storing the full matrix.

(d) Without a cache the total is about $\tfrac{a}{2} n^2 + \tfrac{b}{3} n^3$, cubic at the top. With a KV cache it is about $a n + \tfrac{b}{2} n^2$, quadratic at the top.

</details>